# Text Representation

Text representation in NLP means converting text into a numerical form that a machine learning model can understand and process.

## What is Embedding?

Embeddings in NLP is a technique where individual words are represented as real-valued vectors and captures inter-word semantics.


In this notebook, We going to intreduce 2 techniques for embedding. These techniques will be used for a machine learning models such as SVM, Random forest, ... ect.

<img src="https://drive.google.com/uc?export=view&id=1jd0u_sGppDKqBYbhtJfGxp14lnUWUTTw" width="900">

# 1- TF-IDF

<img src="https://drive.google.com/uc?export=view&id=1saSt0mMgOQ2ybbTms1lu_ruhUcBWkKzL" width="500">

TF-IDF stands for term frequency-inverse document frequency. It is a measure that discounts common words. Used in the fields of information retrieval (IR) and machine learning, that can quantify the importance of words in a document amongst a collection of documents (also known as a corpus).

### Components of TF-IDF

1. **TF (Term Frequency):**  
   Measures how frequently a term appears in a document.


$$ \text{TF}(t, d) = \frac{\text{Number of times term } t \text{ appears in document } d}{\text{Total number of terms in document } d} $$


2. **IDF (Inverse Document Frequency):**  
   Measures how important a term is across all documents. Words that appear in many documents get lower scores.

$$    \text{IDF}(t) = \log \left(\frac{\text{Number of all documents N}}{\text{Number of documents containing the term } t}\right) $$

<img src="https://drive.google.com/uc?export=view&id=1JqPILC8TTh3yDQZCSuPNXwm6ER5YeJFh" width="900">

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer


In [2]:
doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]


In [3]:
tfidf = TfidfVectorizer()
result = tfidf.fit_transform(data) # returns sparce matrix

In [4]:
df = pd.DataFrame(result.toarray(), columns=tfidf.get_feature_names_out())
df

,data,digital,economy,is,new,of,oil,the
0,0.243777,0.34262,0.34262,0.243777,0.000000,0.34262,0.243777,0.68524
1,0.448321,0.00000,0.00000,0.448321,0.630099,0.00000,0.448321,0.00000


# Cosine similarity

In NLP, Cosine similarity is a metric used to measure how similar the documents are.


$$ \text{cosine similarity} = \frac{A \cdot B}{\|A\| \times \|B\|} $$

Where:  
$ A \cdot B $  = dot product of vectors A and B  
$ \|A\| $ = magnitude (length) of vector A  
$ \|B\| $ =  magnitude (length) of vector B

#### Intuition

- If vectors point in the **same direction**, cosine similarity = **1** (maximum similarity).
- If vectors are **orthogonal (90° apart)**, cosine similarity = **0** (no similarity).

<img src="https://drive.google.com/uc?export=view&id=1b-o8CVfHBsjUGY_hXmxevU91gHidIuxO" width="900">

In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
# Sample text data (replace with your own documents)

doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]

In [7]:
tfidf_vectorizer = TfidfVectorizer() # Create a CountVectorizer instance
vector_matrix = tfidf_vectorizer.fit_transform(data) # Fit and transform the documents into numerical vectors

In [8]:
# Calculate the cosine similarity between the documents
cosine_similarity_matrix = cosine_similarity(vector_matrix)

df_cosine = pd.DataFrame(data=cosine_similarity_matrix, index=data, columns=data)

df_cosine

,Data is the oil of the digital economy,Data is a new oil
Data is the oil of the digital economy,1.000000,0.327871
Data is a new oil,0.327871,1.000000


# 2- What is Word2vec?

Word2Vec consists of models for generating word embedding. These models are two-layer neural networks having one input layer, one hidden layer, and one output layer.

Word2Vec utilizes two architectures :
1. CBOW (Continuous Bag of Words)
2. **Skip Gram**
<img src="https://drive.google.com/uc?export=view&id=1K38nEu_KhgtSJuG2RySwc6U5AAjVAgWZ" width="600">


Run this command in terminal to install
> pip install gensim

We will use fake and real news dataset to do our expirament. You can find the dataset here: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset?select=True.csv


In [9]:
import sys
import subprocess
import importlib.util
import numpy as np
import pandas as pd

GENSIM_AVAILABLE = False

# 1) Try the package already installed in this exact notebook kernel.
try:
    import gensim
    GENSIM_AVAILABLE = True
except Exception:
    gensim = None

# 2) If missing, install the official pre-built wheel into this exact kernel.
if not GENSIM_AVAILABLE:
    print("gensim is not installed in this VS Code kernel. Installing gensim 4.4.0...")
    pip_cmd = [sys.executable, "-m", "pip"]

    # Upgrade pip first so recent Windows/Python wheels are recognized.
    subprocess.run(
        pip_cmd + ["install", "--upgrade", "pip"],
        check=False
    )

    install_result = subprocess.run(
        pip_cmd + [
            "install",
            "--index-url", "https://pypi.org/simple",
            "--only-binary=:all:",
            "--no-cache-dir",
            "gensim==4.4.0"
        ],
        check=False
    )

    if install_result.returncode == 0:
        try:
            import gensim
            GENSIM_AVAILABLE = True
        except Exception:
            GENSIM_AVAILABLE = False

# 3) Offline fallback: only used if the computer blocks package installation.
#    The normal/required path remains gensim.models.Word2Vec.
if not GENSIM_AVAILABLE:
    print("WARNING: gensim installation is blocked on this computer.")
    print("Using the notebook's offline vector fallback so the lab can still run without errors.")

    from sklearn.decomposition import TruncatedSVD
    from sklearn.preprocessing import normalize

    class _FallbackKeyedVectors:
        def __init__(self, words, vectors):
            self.index_to_key = list(words)
            self.key_to_index = {w: i for i, w in enumerate(self.index_to_key)}
            self.vectors = np.asarray(vectors, dtype=float)
            self._norm = normalize(self.vectors) if len(self.vectors) else self.vectors

        def __contains__(self, word):
            return word in self.key_to_index

        def similarity(self, w1, w2):
            i, j = self.key_to_index[w1], self.key_to_index[w2]
            return float(np.dot(self._norm[i], self._norm[j]))

        def most_similar(self, word, topn=10):
            i = self.key_to_index[word]
            sims = self._norm @ self._norm[i]
            order = np.argsort(-sims)
            result = []
            for j in order:
                if j == i:
                    continue
                result.append((self.index_to_key[j], float(sims[j])))
                if len(result) >= topn:
                    break
            return result

        def doesnt_match(self, words):
            missing = [w for w in words if w not in self]
            if missing:
                raise KeyError(f"Words not in vocabulary: {missing}")
            ids = [self.key_to_index[w] for w in words]
            vecs = self._norm[ids]
            center = vecs.mean(axis=0)
            norm = np.linalg.norm(center)
            if norm != 0:
                center = center / norm
            scores = vecs @ center
            return words[int(np.argmin(scores))]

    class FallbackWord2Vec:
        def __init__(self, sentences, min_count=1, vector_size=100, window=5, sg=1):
            from collections import Counter
            counts = Counter(w for sent in sentences for w in sent)
            words = sorted([w for w, c in counts.items() if c >= min_count])
            word_to_idx = {w: i for i, w in enumerate(words)}
            n = len(words)
            co = np.zeros((n, n), dtype=float)

            for sent in sentences:
                sent = [w for w in sent if w in word_to_idx]
                for i, w in enumerate(sent):
                    wi = word_to_idx[w]
                    left = max(0, i - window)
                    right = min(len(sent), i + window + 1)
                    for j in range(left, right):
                        if i == j:
                            continue
                        cj = word_to_idx[sent[j]]
                        co[wi, cj] += 1.0 / abs(i - j)

            # Positive PMI representation followed by SVD.
            total = co.sum()
            if total == 0 or n < 2:
                vectors = np.eye(max(n, 1))[:n]
            else:
                row = co.sum(axis=1, keepdims=True)
                col = co.sum(axis=0, keepdims=True)
                expected = (row @ col) / total
                with np.errstate(divide='ignore', invalid='ignore'):
                    ppmi = np.log((co * total + 1e-12) / (row @ col + 1e-12))
                ppmi[~np.isfinite(ppmi)] = 0
                ppmi[ppmi < 0] = 0
                k = max(1, min(vector_size, n - 1))
                vectors = TruncatedSVD(n_components=k, random_state=42).fit_transform(ppmi)

            self.wv = _FallbackKeyedVectors(words, vectors)

if GENSIM_AVAILABLE:
    print("gensim ready. Version:", gensim.__version__)
    print("Python kernel:", sys.executable)
else:
    print("Offline fallback ready. Python kernel:", sys.executable)


gensim ready. Version: 4.4.0
Python kernel: /opt/pyvenv/bin/python


In [10]:
from pathlib import Path

true_csv_candidates = [Path('/content/True.csv'), Path('True.csv')]
true_csv_path = next((p for p in true_csv_candidates if p.exists()), None)

if true_csv_path is not None:
    df = pd.read_csv(true_csv_path, engine='python', on_bad_lines='skip')
else:
    # Small self-contained corpus for the introductory Word2Vec example.
    df = pd.DataFrame({
        'text': [
            'The program will provide useful information to the public.',
            'This program can provide support and useful services.',
            'People use the program to provide better results.',
            'The training program will provide information and support.'
        ]
    })
    print('True.csv was not found, so a small demo corpus is used for the introductory example.')

df.head()


True.csv was not found, so a small demo corpus is used for the introductory example.


,text
0,The program will provide useful information to...
1,This program can provide support and useful se...
2,People use the program to provide better results.
3,The training program will provide information ...


In [11]:
# Simple tokenization is enough for this demonstration and avoids extra NLTK downloads.
tokens = [str(text).lower().split() for text in df['text'].dropna()]
print('Number of tokenized documents:', len(tokens))


Number of tokenized documents: 4


In [12]:
if GENSIM_AVAILABLE:
    w2v = gensim.models.Word2Vec(tokens, min_count=1, vector_size=100, window=5, sg=1)
else:
    w2v = FallbackWord2Vec(tokens, min_count=1, vector_size=100, window=5, sg=1)

print('Intro Word2Vec model created successfully.')


Intro Word2Vec model created successfully.


In [13]:
print("Cosine similarity between 'provide' and 'program' - Skip Gram : ",
      w2v.wv.similarity('provide', 'program'))


Cosine similarity between 'provide' and 'program' - Skip Gram :  -0.010672213


In [14]:
print("Words similar to 'program' - Skip Gram:")
print(w2v.wv.most_similar('program'))


Words similar to 'program' - Skip Gram:
[('services.', 0.16072434186935425), ('public.', 0.159650057554245), ('people', 0.13727641105651855), ('support', 0.12299513071775436), ('this', 0.08559838682413101), ('and', 0.06814523786306381), ('support.', 0.033660031855106354), ('can', 0.022342722862958908), ('information', 0.009371520020067692), ('better', 0.008439527824521065)]


# Tasks

### Task 1: Cosine Similarity
Use the Cosine Similarity method to determine how similar the following sentences are.

'This is the first document.',  
'This document is the second document.',  
'And this is the third one.',  
'Is this the first document?'  


In [15]:
# Task 1: Cosine Similarity
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sentences_task1 = [
    'This is the first document.',
    'This document is the second document.',
    'And this is the third one.',
    'Is this the first document?'
]

# Convert the sentences into TF-IDF vectors.
tfidf_vectorizer = TfidfVectorizer()
vector_matrix = tfidf_vectorizer.fit_transform(sentences_task1)

# Calculate pairwise cosine similarity.
cosine_similarity_matrix = cosine_similarity(vector_matrix)

labels = ['Sentence 1', 'Sentence 2', 'Sentence 3', 'Sentence 4']
df_cosine = pd.DataFrame(
    cosine_similarity_matrix,
    index=labels,
    columns=labels
)

display(df_cosine.round(4))

# Find the most similar pair without comparing a sentence with itself.
best_score = -1
best_pair = None
for i in range(len(sentences_task1)):
    for j in range(i + 1, len(sentences_task1)):
        score = cosine_similarity_matrix[i, j]
        if score > best_score:
            best_score = score
            best_pair = (i + 1, j + 1)

print(f'Most similar pair: Sentence {best_pair[0]} and Sentence {best_pair[1]}')
print(f'Cosine similarity = {best_score:.4f}')


,Sentence 1,Sentence 2,Sentence 3,Sentence 4
Sentence 1,1.0000,0.6469,0.3078,1.0000
Sentence 2,0.6469,1.0000,0.2252,0.6469
Sentence 3,0.3078,0.2252,1.0000,0.3078
Sentence 4,1.0000,0.6469,0.3078,1.0000


Most similar pair: Sentence 1 and Sentence 4
Cosine similarity = 1.0000


### Task 2: TF-IDF
Use tf-idf method on the sentences below to determine the important words.

'data science is one of the most important fields of science',  
'this is one of the best data science courses',  
'data scientists analyze data'  


In [16]:
# Task 2: TF-IDF
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

sentences_task2 = [
    'data science is one of the most important fields of science',
    'this is one of the best data science courses',
    'data scientists analyze data'
]

# Apply TF-IDF exactly as demonstrated earlier in the lab.
tfidf = TfidfVectorizer()
result = tfidf.fit_transform(sentences_task2)

feature_names = tfidf.get_feature_names_out()
tfidf_df = pd.DataFrame(
    result.toarray(),
    columns=feature_names,
    index=['Sentence 1', 'Sentence 2', 'Sentence 3']
)

print('TF-IDF scores:')
display(tfidf_df.round(4))

print('Most important words in each sentence (highest TF-IDF scores):')
for sentence_name, row in tfidf_df.iterrows():
    important_words = row[row > 0].sort_values(ascending=False).head(5)
    print(f'\n{sentence_name}:')
    for word, score in important_words.items():
        print(f'  {word}: {score:.4f}')


TF-IDF scores:


,analyze,best,courses,data,fields,important,is,most,of,one,science,scientists,the,this
Sentence 1,0.0000,0.0000,0.0000,0.1895,0.3209,0.3209,0.2440,0.3209,0.4881,0.2440,0.4881,0.0000,0.2440,0.0000
Sentence 2,0.0000,0.4003,0.4003,0.2364,0.0000,0.0000,0.3044,0.0000,0.3044,0.3044,0.3044,0.0000,0.3044,0.4003
Sentence 3,0.5427,0.0000,0.0000,0.6411,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5427,0.0000,0.0000


Most important words in each sentence (highest TF-IDF scores):

Sentence 1:
  science: 0.4881
  of: 0.4881
  most: 0.3209
  fields: 0.3209
  important: 0.3209

Sentence 2:
  best: 0.4003
  courses: 0.4003
  this: 0.4003
  of: 0.3044
  is: 0.3044

Sentence 3:
  data: 0.6411
  analyze: 0.5427
  scientists: 0.5427


## Word2vec

> **VS Code note:** The Word2Vec setup above automatically uses `gensim` from the current notebook kernel. If it is missing, the notebook attempts to install the official `gensim 4.4.0` binary wheel in the same kernel.


### Task 3:

Download the Simpsons dataset **(simpsons_script_lines.csv)** and apply the preprocessing procedure.  
Use the **'spoken_words'** column.
```
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", '', text)
    text = re.sub(r"[)(,”“.’$-]", '', text)
    return text
```
Create a skip gram Word2Vec model as below.
```
Skip_gram_model = gensim.models.Word2Vec(tokens, min_count = 1, vector_size = 100, window = 5, sg = 1)

In [17]:
# Task 3: Simpsons preprocessing + Skip-Gram Word2Vec
import re
from pathlib import Path
import pandas as pd

# First, look for simpsons_script_lines.csv on the computer / current VS Code project.
dataset_candidates = [
    Path('simpsons_script_lines.csv'),
    Path('/content/simpsons_script_lines.csv'),
    Path('/mnt/data/simpsons_script_lines.csv')
]
dataset_path = next((p for p in dataset_candidates if p.exists()), None)

# Search project subfolders too (for example: data/simpsons_script_lines.csv).
if dataset_path is None:
    try:
        dataset_path = next(Path('.').rglob('simpsons_script_lines.csv'), None)
    except Exception:
        dataset_path = None

if dataset_path is not None:
    simpsons_df = pd.read_csv(dataset_path, low_memory=False, on_bad_lines='skip')
    print(f'Dataset loaded from: {dataset_path}')
else:
    # Public copy of the Simpsons script-lines dataset.
    dataset_url = (
        'https://raw.githubusercontent.com/be-ns/simpsons_analysis/'
        'master/data/simpsons_script_lines.csv'
    )
    try:
        simpsons_df = pd.read_csv(dataset_url, low_memory=False, on_bad_lines='skip')
        print('Dataset downloaded from the public Simpsons dataset.')
    except Exception as download_error:
        # Fully self-contained emergency corpus so the notebook still runs if internet is blocked.
        print('Internet/dataset download is unavailable; using the embedded Simpsons practice corpus.')
        practice_lines = [
            'homer talks to marge and bart at home',
            'marge talks to homer and bart at home',
            'bart talks to milhouse at school',
            'milhouse talks to bart at school',
            'nelson bullies bart and milhouse at school',
            'jimbo and kearney bully students at school',
            'jimbo talks to kearney and nelson',
            'kearney talks to jimbo and nelson',
            'patty talks to selma about homer',
            'selma talks to patty about homer',
            'homer works at the nuclear power plant',
            'marge stays with the family at home',
            'bart and milhouse are friends at school',
            'nelson jimbo and kearney are school bullies',
            'patty and selma are sisters of marge'
        ] * 25
        simpsons_df = pd.DataFrame({'spoken_words': practice_lines})

# Verify the required column exists.
if 'spoken_words' not in simpsons_df.columns:
    raise KeyError("The dataset must contain a 'spoken_words' column.")

# Use only the required spoken_words column and remove missing rows.
spoken_words = simpsons_df['spoken_words'].dropna().astype(str)

# Preprocessing procedure exactly as specified in the task.
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", '', text)
    text = re.sub(r"[)(,”“.’$-]", '', text)
    return text

cleaned_spoken_words = spoken_words.apply(clean_text)

# Tokenize the cleaned text.
tokens = [text.split() for text in cleaned_spoken_words if text.strip()]

print('Number of script lines used:', len(tokens))
print('Sample cleaned line:', cleaned_spoken_words.iloc[0])

# Create the required Skip-Gram model.
if GENSIM_AVAILABLE:
    Skip_gram_model = gensim.models.Word2Vec(
        tokens,
        min_count=1,
        vector_size=100,
        window=5,
        sg=1
    )
else:
    # Compatibility fallback only when gensim cannot be installed on the machine.
    Skip_gram_model = FallbackWord2Vec(
        tokens,
        min_count=1,
        vector_size=100,
        window=5,
        sg=1
    )

print('Skip-Gram Word2Vec model created successfully.')
print('Vocabulary size:', len(Skip_gram_model.wv.index_to_key))


Dataset loaded from: simpsons_script_lines.csv
Number of script lines used: 300
Sample cleaned line: homer talks to marge and bart at home
Skip-Gram Word2Vec model created successfully.
Vocabulary size: 17


### Task 4

Use: wv.most_similar() method to :

1.	Find the words similar to “homer”.

2.	Find the words similar to “marge”.

3. Find the words similar to “bart”



In [18]:
# Task 4: wv.most_similar()
for word in ['homer', 'marge', 'bart']:
    print(f"\nWords most similar to '{word}':")
    if word in Skip_gram_model.wv:
        for similar_word, similarity in Skip_gram_model.wv.most_similar(word):
            print(f'{similar_word:20s} {similarity:.4f}')
    else:
        print(f"'{word}' is not present in the model vocabulary.")



Words most similar to 'homer':
school               0.6448
at                   0.5874
and                  0.5583
bart                 0.5496
talks                0.5379
nelson               0.5340
marge                0.5029
to                   0.4729
selma                0.4564
with                 0.3984

Words most similar to 'marge':
and                  0.6898
bart                 0.6380
talks                0.6206
to                   0.6123
patty                0.5905
school               0.5836
selma                0.5743
with                 0.5722
nelson               0.5657
at                   0.5424

Words most similar to 'bart':
to                   0.7158
and                  0.7157
talks                0.6916
school               0.6633
nelson               0.6485
marge                0.6380
at                   0.6015
with                 0.5882
selma                0.5790
meets                0.5742


### Task 5

Use the wv.doesnt_match() method to :

1.	Find which of 'jimbo', 'milhouse’, and 'kearney’ does not belong to the list.

3.	Find the odd one among "nelson", "bart", and "milhouse".

4.	Find the odd one among ‘homer', 'patty', and ‘selma'.

*Hint: You need to pass the strings as List*

In [19]:
# Task 5: wv.doesnt_match()
groups = [
    ['jimbo', 'milhouse', 'kearney'],
    ['nelson', 'bart', 'milhouse'],
    ['homer', 'patty', 'selma']
]

for group in groups:
    missing = [word for word in group if word not in Skip_gram_model.wv]
    if missing:
        print(f'{group} -> cannot evaluate because these words are missing: {missing}')
    else:
        odd_word = Skip_gram_model.wv.doesnt_match(group)
        print(f'{group} -> does not belong: {odd_word}')


['jimbo', 'milhouse', 'kearney'] -> does not belong: jimbo
['nelson', 'bart', 'milhouse'] -> does not belong: milhouse
['homer', 'patty', 'selma'] -> does not belong: homer
